In [1]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

print("device_count:", torch.cuda.device_count())
print("current_device:", torch.cuda.current_device())
print("device_name(0):", torch.cuda.get_device_name(0))
print("device_string:", torch.device("cuda:0"))

2.3.1
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce GTX 1650
device_count: 1
current_device: 0
device_name(0): NVIDIA GeForce GTX 1650
device_string: cuda:0


In [2]:
import pandas as pd

s_data = pd.read_parquet('all_samples.parquet')
data = s_data.loc[:, ['ecg_raw', 'label']].copy()

bad2good_idx = data[data['label']=='bad2good']['label'].index
data.loc[bad2good_idx, ['label']] = 'good'

label_map = {'good':0, 'bad': 1}
data['label_int'] = data['label'].map(label_map)

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

def ecg_transform(sample):
    x = sample["data"].astype(np.float32)          # (2500,)
    # per-sample z-score
    m = x.mean()
    s = x.std()
    x = (x - m) / (s + 1e-8)
    # Conv1d expects (C, L)
    x = torch.from_numpy(x).unsqueeze(0)           # (1, 2500)
    y = torch.tensor(sample["labels"], dtype=torch.long)
    return {"data": x, "labels": y}

class ECGDataset(Dataset):
    def __init__(self, df, x_col="ecg_raw", y_col="label_int", transform=None):
        self.df = df.reset_index(drop=True)
        self.x_col = x_col
        self.y_col = y_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        sample = {
            "data": np.asarray(self.df.at[index, self.x_col]).copy(),
            "labels": int(self.df.at[index, self.y_col]),
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

# Dataset
ds = ECGDataset(data, transform=ecg_transform)

# 用 sampler 解决不平衡
labels = data["label_int"].to_numpy()
class_counts = np.bincount(labels)                 # [good, bad]
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels]
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(labels),
    replacement=True,
)

loader = DataLoader(ds, batch_size=64, sampler=sampler, num_workers=0)

batch = next(iter(loader))
print(batch["data"].shape, batch["labels"].shape)  # (64, 1, 2500) (64,)

torch.Size([64, 1, 2500]) torch.Size([64])


In [12]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch

idx = data.index.to_numpy()
y = data["label_int"].to_numpy()

# 1) 先切 train vs test（分层）
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=2026, stratify=y
)

# 2) 再从 train 切出 val（分层）
train_y = data.loc[train_idx, "label_int"].to_numpy()
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.1, random_state=2026, stratify=train_y
)

# 3) Dataset
train_ds = ECGDataset(data.loc[train_idx].reset_index(drop=True))
val_ds   = ECGDataset(data.loc[val_idx].reset_index(drop=True))
test_ds  = ECGDataset(data.loc[test_idx].reset_index(drop=True))

# 4) 只给 train 做 sampler（基于 train labels）
train_labels = data.loc[train_idx, "label_int"].to_numpy()
class_counts = np.bincount(train_labels)          # [good, bad]
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

train_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_labels),
    replacement=True
)

# 5) DataLoader：train 用 sampler；val/test 不用
train_loader = DataLoader(train_ds, batch_size=64, sampler=train_sampler, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0)

print(len(train_ds), len(val_ds), len(test_ds))
batch = next(iter(train_loader))
print(batch["data"].shape, batch["labels"].shape)

21172 2353 5882
torch.Size([64, 2500]) torch.Size([64])


In [14]:
batch['data']

tensor([[359, 358, 363,  ..., 322, 320, 327],
        [328, 326, 326,  ..., 364, 363, 362],
        [324, 322, 320,  ..., 318, 320, 319],
        ...,
        [325, 324, 323,  ..., 321, 320, 320],
        [350, 352, 350,  ..., 322, 322, 321],
        [324, 326, 327,  ..., 317, 312, 311]])

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 7, padding=3)
        self.mp = nn.MaxPool1d(5)
        self.conv2 = nn.Conv1d(32, 32, 7, padding=3)
        self.gap = nn.AdaptiveAvgPool1d(1)   # (B, 32, 1)
        self.linear1 = nn.Linear(32, 128)
        self.linear2 = nn.Linear(128, 1)     # binary logit

    def forward(self, x):
        # x: (B, 1, L)
        x = self.mp(torch.relu(self.conv1(x)))
        x = torch.relu(self.conv2(x))
        x = self.gap(x).squeeze(-1)          # (B, 32)
        x = torch.relu(self.linear1(x))
        x = self.linear2(x).squeeze(-1)      # (B,)
        return x


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ConvNet().to(device)

criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [21]:
def accuracy_from_logits(logits, y):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).long()
    return (preds == y.long()).float().mean().item()

def train_one_epoch(dataloader, model, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n = 0

    for batch in dataloader:
        x = batch["data"].to(device).float()  # (B, 2500)
        if x.ndim == 2:
            x = x.unsqueeze(1)   # (B, 1, 2500)
        y = batch["labels"].to(device).float() # (B,) float for BCE

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits.detach(), y.detach()) * bs
        n += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate(dataloader, model, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n = 0

    for batch in dataloader:
        x = batch["data"].to(device).float()  # (B, 2500)
        if x.ndim == 2:
            x = x.unsqueeze(1)   # (B, 1, 2500)
        y = batch["labels"].to(device).float() # (B,) float for BCE

        logits = model(x)
        loss = criterion(logits, y)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        n += bs

    return total_loss / n, total_acc / n

In [22]:
epochs = 50
best_vloss = float("inf")
history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(train_loader, model, criterion, optimizer, device)
    val_loss, val_acc = evaluate(val_loader, model, criterion, device)

    if val_loss < best_vloss:
        best_vloss = val_loss
        torch.save(model.state_dict(), "model.pt")

    history['loss'].append(train_loss)
    history['acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch:03d}: LOSS train {train_loss:.3f} valid {val_loss:.3f}; "
          f"ACC train {train_acc:.4f} valid {val_acc:.4f}")

Epoch 000: LOSS train 0.460 valid 0.501; ACC train 0.8406 valid 0.9184
Epoch 001: LOSS train 0.428 valid 0.275; ACC train 0.8577 valid 0.9545
Epoch 002: LOSS train 0.396 valid 0.217; ACC train 0.8704 valid 0.9618
Epoch 003: LOSS train 0.363 valid 0.241; ACC train 0.8823 valid 0.9558
Epoch 004: LOSS train 0.346 valid 0.244; ACC train 0.8908 valid 0.9567
Epoch 005: LOSS train 0.315 valid 0.140; ACC train 0.8987 valid 0.9783
Epoch 006: LOSS train 0.309 valid 0.228; ACC train 0.9001 valid 0.9647
Epoch 007: LOSS train 0.294 valid 0.215; ACC train 0.9044 valid 0.9720
Epoch 008: LOSS train 0.286 valid 0.293; ACC train 0.9082 valid 0.9605
Epoch 009: LOSS train 0.265 valid 0.208; ACC train 0.9128 valid 0.9800
Epoch 010: LOSS train 0.250 valid 0.118; ACC train 0.9206 valid 0.9856
Epoch 011: LOSS train 0.230 valid 0.250; ACC train 0.9251 valid 0.9690
Epoch 012: LOSS train 0.218 valid 0.186; ACC train 0.9243 valid 0.9762
Epoch 013: LOSS train 0.205 valid 0.147; ACC train 0.9291 valid 0.9839
Epoch 

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load("model.pt", map_location=device))
test_loss, test_acc = evaluate(test_dataloader, model, criterion, device)
print(f"TEST: loss {test_loss:.3f}, acc {test_acc:.4f}")